# Cross-Dataset Evaluation: SavannaTreeAI

**Dataset**: SavannaTreeAI (Jansen et al., 2023)  
**Biome**: Savanna woodland (Northern Australia)  
**GSD**: 1.7–2 cm  
**Samples**: 7 sites, 2,547 polygons, 36 species  
**Format**: COCO instance segmentation (local, requires pycocotools)

**Model**: OneFormer Full-Res (seed 42, trained on OAM-TCD)

In [ ]:
# ============================================================================
# Setup
# ============================================================================
import sys, os, json, platform
from pathlib import Path
from typing import Dict, Optional

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

IS_HPC = platform.system() != "Darwin"
if IS_HPC:
    HPC_BASE = Path("/home/awilaiwo/Data/BARE_Related/BARE_Journal")
    ROOT_DIR = HPC_BASE
    NOTEBOOK_DIR = HPC_BASE / "Notebook"
    DATASET_BANK_DIR = HPC_BASE / "Dataset_Bank"
    
    # On HPC, Core and Extended are directly under BARE_Journal based on screenshot
    CODEBASE_DIR = HPC_BASE
else:
    ROOT_DIR = Path(os.path.abspath("")).parent.parent
    NOTEBOOK_DIR = ROOT_DIR / "Codebase" / "Notebook"
    DATASET_BANK_DIR = ROOT_DIR / "Dataset_Bank"
    CODEBASE_DIR = ROOT_DIR / "Codebase"

EXPORT_DIR = NOTEBOOK_DIR / "exports"
EXPORT_DIR.mkdir(exist_ok=True)

core_dir = str(CODEBASE_DIR / "Core")
extended_dir = str(CODEBASE_DIR / "Extended")

sys.path.insert(0, core_dir)
sys.path.insert(0, extended_dir)

print(f"Environment: {'UTS iHPC' if IS_HPC else 'Local Mac'}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\n--- Diagnostic Paths ---")
print(f"Core directory path: {core_dir}")
print(f"Core directory exists: {os.path.exists(core_dir)}")
print(f"Extended directory path: {extended_dir}")
print(f"Extended directory exists: {os.path.exists(extended_dir)}")
if os.path.exists(core_dir):
    print(f"checkpoint.py exists in Core: {os.path.exists(os.path.join(core_dir, 'checkpoint.py'))}")

In [ ]:
# ============================================================================
# Configuration — SET THESE BEFORE RUNNING
# ============================================================================

# Path to best OneFormer Full-Res checkpoint (seed 42)
if IS_HPC:
    MODEL_PATH = str(HPC_BASE / "Notebook" / "oneformer_fullres_42" / "Notebook" / "outputs_oneformer_fullres_42" / "best_checkpoint")
else:
    MODEL_PATH = str(ROOT_DIR / "Experiment_Repo" / "OneFormer" / "oneformer_fullres_42" / "Notebook" / "outputs_oneformer_fullres_42" / "best_checkpoint")

# SavannaTreeAI local path
# Expected structure: root/images/*.png + root/annotations/annotations.json (COCO)
DATASET_ROOT = str(DATASET_BANK_DIR / "SavannaTreeAI")
DATASET_NAME = "savannatreeai"
DATASET_SPLIT = "val"  # Use val split for evaluation

BATCH_SIZE = 1
NUM_WORKERS = 8

assert MODEL_PATH is not None, "Set MODEL_PATH to your OneFormer Full-Res checkpoint"
print(f"Dataset root: {DATASET_ROOT} (exists: {Path(DATASET_ROOT).exists()})")
print(f"Model path: {MODEL_PATH}")

In [ ]:
# ============================================================================
# Load Model & Dataset
# ============================================================================
from checkpoint import load_model_for_evaluation
from dataset import ExternalDatasetAdapter
from datasets_loader import load_tree_dataset
from metrics import calculate_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading model from {MODEL_PATH}...")
model, image_processor = load_model_for_evaluation(
    model_path=MODEL_PATH, config=None, device=device)
model = model.to(device)
model.eval()
print(f"Model loaded: {model.__class__.__name__}")

print(f"\nLoading SavannaTreeAI from {DATASET_ROOT} (split={DATASET_SPLIT})...")
ann_file = str(Path(DATASET_ROOT) / "annotations" / "val" / "instancesvalidation.json")
print(f"Annotation file: {ann_file} (exists: {Path(ann_file).exists()})")
ext_dataset = load_tree_dataset(
    dataset_name=DATASET_NAME, root_dir=DATASET_ROOT, split=DATASET_SPLIT,
    annotation_file=ann_file)
print(f"Loaded {len(ext_dataset)} samples")

adapted = ExternalDatasetAdapter(
    external_dataset=ext_dataset, image_processor=image_processor,
    target_size=(1024, 1024))
eval_loader = DataLoader(
    adapted, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=False)
print(f"DataLoader: {len(eval_loader)} batches")

In [ ]:
# ============================================================================
# Run Evaluation
# ============================================================================
all_preds, all_labels = [], []

with torch.no_grad():
    for batch_idx, batch in enumerate(eval_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        outputs = model(pixel_values=pixel_values)
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs

        target_h, target_w = labels.shape[-2:]
        if logits.shape[-2:] != (target_h, target_w):
            logits = F.interpolate(logits, size=(target_h, target_w),
                                   mode="bilinear", align_corners=False)

        all_preds.append(logits.argmax(dim=1).cpu().numpy().astype(np.uint8))
        all_labels.append(labels.numpy().astype(np.uint8))

        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx + 1}/{len(eval_loader)}")

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)
print(f"Evaluation complete: {all_preds.shape[0]} samples, dtype={all_preds.dtype}")

In [ ]:
# ============================================================================
# Calculate & Display Metrics
# ============================================================================
metrics = calculate_metrics(
    preds=all_preds, labels=all_labels, ignore_index=255,
    metrics_list=['pixel_accuracy', 'iou_class_1', 'dice_class_1',
                  'precision_class_1', 'recall_class_1',
                  'f1_score_class_1', 'boundary_iou_class_1'])

print(f"{'='*50}")
print(f"SavannaTreeAI Results (OneFormer Full-Res, zero-shot)")
print(f"{'='*50}")
for k, v in sorted(metrics.items()):
    print(f"  {k}: {v:.4f}")

# Save results
results_path = EXPORT_DIR / "cross_eval_savannatreeai.json"
with open(results_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nSaved to {results_path}")

In [ ]:
# ============================================================================
# Sample Prediction Visualization
# ============================================================================
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation, generate_binary_structure

VIZ_DIR = EXPORT_DIR / "visualizations"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

NUM_VIZ = 3
TREE_COLOR = np.array([0.0, 0.8, 0.0, 0.55])
IGNORE_COLOR = np.array([0.5, 0.5, 0.5, 0.3])

# Boundary extraction helper (matches metrics.py logic but thinner for visualization)
BOUNDARY_DILATION_RATIO = 0.005  # ~7px on 1024x1024 (thinner than metrics' 0.02)

def extract_boundary(mask, dilation_ratio=BOUNDARY_DILATION_RATIO):
    h, w = mask.shape
    dilation_pixels = int(max(1, np.sqrt(h**2 + w**2) * dilation_ratio))
    struct = generate_binary_structure(2, 1)
    if mask.any() and not mask.all():
        dilated = binary_dilation(mask, structure=struct, iterations=dilation_pixels)
        return dilated ^ mask
    return np.zeros_like(mask, dtype=bool)

# Find non-empty samples (cast to int for HuggingFace compatibility)
candidates = [int(x) for x in np.random.permutation(all_preds.shape[0])]
selected = []
for i in candidates:
    img_sample = adapted[i]
    pv = img_sample['pixel_values'].numpy()
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    img_check = (pv * std + mean).clip(0, 1)
    if img_check.mean() > 0.05:
        selected.append(i)
    if len(selected) >= NUM_VIZ:
        break

fig, axes = plt.subplots(len(selected), 5, figsize=(30, 6 * len(selected)))
if len(selected) == 1:
    axes = axes.reshape(1, -1)

for row, si in enumerate(selected):
    sample = adapted[si]
    pv = sample['pixel_values'].numpy()
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    img_np = np.transpose((pv * std + mean).clip(0, 1), (1, 2, 0))

    pred = all_preds[si]
    label = all_labels[si]
    valid = label != 255

    # Col 1: Original
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f'Sample {si}: Original', fontsize=12, fontweight='bold')
    axes[row, 0].axis('off')

    # Col 2: Ground Truth overlay
    gt_overlay = np.zeros((*label.shape, 4))
    gt_overlay[label == 1] = TREE_COLOR
    gt_overlay[label == 255] = IGNORE_COLOR
    axes[row, 1].imshow(img_np)
    axes[row, 1].imshow(gt_overlay)
    axes[row, 1].set_title('Ground Truth\n(Green=Tree, Gray=Ignore)', fontsize=12, fontweight='bold')
    axes[row, 1].axis('off')

    # Col 3: Prediction overlay
    pred_overlay = np.zeros((*pred.shape, 4))
    pred_overlay[pred == 1] = TREE_COLOR
    axes[row, 2].imshow(img_np)
    axes[row, 2].imshow(pred_overlay)
    axes[row, 2].set_title('Prediction\n(Green=Predicted Tree)', fontsize=12, fontweight='bold')
    axes[row, 2].axis('off')

    # Col 4: Pixel Error Map overlaid on image (Green=TP, Yellow=FP, Red=FN)
    tp = (pred == 1) & (label == 1) & valid
    fp = (pred == 1) & (label == 0) & valid
    fn = (pred == 0) & (label == 1) & valid

    err_overlay = np.zeros((*pred.shape, 4))
    err_overlay[tp] = [0.2, 0.8, 0.2, 0.5]    # Green, semi-transparent
    err_overlay[fp] = [1.0, 0.85, 0.0, 0.6]   # Yellow
    err_overlay[fn] = [0.9, 0.1, 0.1, 0.6]    # Red
    err_overlay[~valid] = [0.3, 0.3, 0.3, 0.4] # Ignore

    axes[row, 3].imshow(img_np)
    axes[row, 3].imshow(err_overlay)
    axes[row, 3].set_title('Pixel Error Map\nGreen=TP  Yellow=FP  Red=FN', fontsize=11, fontweight='bold')
    axes[row, 3].axis('off')

    # Col 5: Boundary IoU Error Map overlaid on image (Green=TP, Yellow=FP, Red=FN)
    gt_mask = (label == 1) & valid
    pred_mask = (pred == 1) & valid
    gt_boundary = extract_boundary(gt_mask)
    pred_boundary = extract_boundary(pred_mask)

    b_tp = gt_boundary & pred_boundary
    b_fp = pred_boundary & ~gt_boundary
    b_fn = gt_boundary & ~pred_boundary

    b_overlay = np.zeros((*pred.shape, 4))
    b_overlay[b_tp] = [0.2, 0.8, 0.2, 0.9]   # Green
    b_overlay[b_fp] = [1.0, 0.85, 0.0, 0.9]  # Yellow
    b_overlay[b_fn] = [0.9, 0.1, 0.1, 0.9]   # Red

    axes[row, 4].imshow(img_np)
    axes[row, 4].imshow(b_overlay)
    axes[row, 4].set_title('Boundary IoU Error Map\nGreen=TP  Yellow=FP  Red=FN', fontsize=11, fontweight='bold')
    axes[row, 4].axis('off')

    s_iou = tp.sum() / (tp.sum() + fp.sum() + fn.sum() + 1e-8)
    b_inter = b_tp.sum()
    b_union = b_tp.sum() + b_fp.sum() + b_fn.sum()
    s_biou = b_inter / (b_union + 1e-8)
    print(f"  Sample {si}: IoU={s_iou:.3f}, B-IoU={s_biou:.3f}")

plt.tight_layout()
viz_path = VIZ_DIR / "savannatreeai_predictions.png"
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
print(f"\nSaved to {viz_path}")
plt.show()